# M08 – Intro File I/O and Simple CLI Patterns

Uses PCEP fundamentals to build small, file-based tools.

---

## Learning Outcomes

- Read/write text files with open() and with.
- Use file methods: read, readline, readlines, write.
- Persist data as JSON; handle errors.
- Structure a CLI with main() loop.

---

## Table of Contents

1. open() and the with statement
2. File methods: read, readline, readlines
3. File methods: write, writelines
4. Encoding (utf-8)
5. pathlib.Path (brief)
6. json module: loads, dumps, load, dump
7. Error handling for file and JSON
8. CLI entrypoint pattern
9. Practice

---

## 1. open() and the with statement

- **open(file, mode='r', encoding=None)** – opens a file. Returns a file object.
- **Modes**: "r" read (default), "w" write (overwrite), "a" append, "x" exclusive create. Add "b" for binary.
- **with open(...) as f:** – ensures the file is closed when the block exits (even on exception). Always use **with** for files.
- **encoding** – use **encoding='utf-8'** for text files to avoid platform-dependent behavior.

In [ ]:
# Pattern: read entire file
# with open("file.txt", "r", encoding="utf-8") as f:
#     content = f.read()

# Pattern: write
# with open("out.txt", "w", encoding="utf-8") as f:
#     f.write("Hello\\n")

# Without with you must call f.close() manually

---

## 2. File methods: read, readline, readlines

| Method | Description | Returns |
|--------|-------------|--------|
| **read()** | Read entire file (or up to size chars if given) | str |
| **readline()** | Read one line (including \\n); empty string at EOF | str |
| **readlines()** | Read all lines as a list of strings (each may end with \\n) | list |

- **for line in f:** – iterate over lines without loading all into memory.

In [ ]:
# Example: count lines (simulated)
content = "line1\\nline2\\nline3\\n"
lines = content.splitlines()
print("Number of lines:", len(lines))

---

## 3. File methods: write, writelines

| Method | Description |
|--------|-------------|
| **write(s)** | Write string s (no automatic newline) |
| **writelines(lines)** | Write each string in iterable (no automatic newlines) |

- Add \\n yourself when writing lines.

---

## 4. Encoding

- **encoding='utf-8'** – standard for text files. Avoids errors on Windows (default cp1252) and ensures consistent behavior.
- **open(path, 'r', encoding='utf-8')** for reading; same for writing.
- If a file is not UTF-8, use the correct encoding or handle UnicodeDecodeError.

---

## 5. pathlib.Path (brief)

- **Path(path)** – object representing a path. Cross-platform.
- **path.exists()** – True if file/dir exists.
- **path.read_text(encoding='utf-8')** – read entire file as string.
- **path.write_text(text, encoding='utf-8')** – write string to file.
- **path.parent** – parent directory. **path.parent.mkdir(parents=True, exist_ok=True)** – create dirs if needed.

In [ ]:
from pathlib import Path
p = Path("data/sample.txt")
print("parent:", p.parent)
# p.parent.mkdir(parents=True, exist_ok=True)
# p.write_text("Hello", encoding="utf-8")

---

## 6. json module: loads, dumps, load, dump

| Function | Purpose |
|----------|--------|
| **json.loads(s)** | Parse JSON string -> Python object (dict, list, etc.) |
| **json.dumps(obj)** | Python object -> JSON string |
| **json.load(file)** | Read from file object and parse |
| **json.dump(obj, file)** | Write obj to file as JSON |

- **json.dumps(obj, indent=2)** – pretty-print. Use try/except for **json.JSONDecodeError** and **OSError** when loading.

In [ ]:
import json
data = {"name": "Alice", "score": 85}
encoded = json.dumps(data)
print(encoded)
decoded = json.loads(encoded)
print(decoded["name"])
print(json.dumps(data, indent=2))

---

## 7. Error handling for file and JSON

- **FileNotFoundError** – file does not exist.
- **PermissionError** – no permission.
- **json.JSONDecodeError** – invalid JSON.
- Pattern: try to load; except use default or log and re-raise.

In [ ]:
def load_json_safe(path: str, default=None):
    default = default if default is not None else {}
    p = Path(path)
    if not p.exists():
        return default
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return default

# Example: load_json_safe("config.json", {})

---

## 8. CLI entrypoint pattern

- One **main()** that runs a **while True** loop (or similar).
- **input(prompt)** for user input; **.strip().lower()** to normalize.
- Dispatch on command (if/elif); **break** to exit.
- Keep file I/O and business logic in separate functions/modules where possible.

In [ ]:
def main() -> None:
    while True:
        cmd = input("Command (add/list/quit): ").strip().lower()
        if cmd == "quit":
            break
        if cmd == "add":
            print("Add logic here")
        elif cmd == "list":
            print("List logic here")
        else:
            print("Unknown command.")

if __name__ == "__main__":
    main()

---

## 9. Practice

1. Write a function that returns the number of lines in a text file (use with open and a loop or readlines).
2. Use Path.read_text and json.loads to load a JSON file; return a default dict if the file is missing or invalid.
3. Use Path.write_text and json.dumps to save a dict to a JSON file with indent=2.
4. Implement a tiny CLI: commands save (write one line to a file), load (print file content), quit.

In [ ]:
# Practice 1: count lines
def count_lines(path: str) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

# Or: len(f.readlines())

In [ ]:
# Practice 2: load JSON safe (see load_json_safe above)

In [ ]:
# Practice 3: save JSON
# Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")

---

## More Examples

**Example: Reading lines one by one (simulated with a string)**

In [ ]:
# Simulate file content
content = "line1\nline2\nline3\n"
for line in content.splitlines():
    print("Line:", repr(line))

**Example: json.dumps with indent and sort_keys**

In [ ]:
import json
config = {"name": "app", "version": 1, "debug": False}
print(json.dumps(config, indent=2, sort_keys=True))

---

## More Practice

**Practice 4:** Write a function that takes a path and returns True if the file exists, False otherwise. Use Path(path).exists().

In [ ]:
from pathlib import Path

def file_exists(path: str) -> bool:
    return Path(path).exists()

print(file_exists("M08_Concepts.ipynb"))  # adjust path if needed
print(file_exists("nonexistent.txt"))

**Practice 5:** Load a JSON string '{"a": 1, "b": 2}' with json.loads. Then add a key "c" with value 3 and convert back to JSON string with json.dumps.

In [ ]:
s = '{"a": 1, "b": 2}'
data = json.loads(s)
data["c"] = 3
print(json.dumps(data))

**Practice 6:** Simulate a two-command CLI: "len" prints the length of a hard-coded string, "upper" prints it uppercased, "quit" exits. Use a loop and strip().lower().

In [ ]:
# Simulated commands (no actual input in notebook)
text = "Hello World"
commands = ["len", "upper", "quit"]
for cmd in commands:
    c = cmd.strip().lower()
    if c == "quit":
        print("Bye")
        break
    if c == "len":
        print(len(text))
    elif c == "upper":
        print(text.upper())